# Step 3 — Metrics

Written and tested **before** any real run, so the pilot and the final run are scored by identical code.
A parse failure (`None`) always counts as wrong.

| Metric | Function |
|---|---|
| Balanced accuracy + 95% bootstrap CI | `balanced_accuracy_ci` |
| Paired bootstrap difference between two judges | `paired_bootstrap` |
| Cohen's κ vs human labels | `cohen_kappa` |
| Hallucination recall; precision reweighted to 10% / 20% prevalence | `recall`, `precision_at_prevalence` |
| Parse-failure rate | `parse_failure_rate` |
| Pairwise: accuracy with both orders, position consistency, share choosing first | `pairwise_outcomes` |
| Cost per 1,000 judgements | `cost_per_1000`, `gpu_cost_per_1000` |

In [1]:
# --- Setup: mount Drive, go to the project, install, import ---------------------------
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/LLM_as_Judge_Benchmark'
REPO_URL = 'https://github.com/moghbaie/LLM_as_Judge_Benchmark.git'

import os, sys, subprocess
if not os.path.exists(PROJECT_DIR):
    subprocess.run(['git', 'clone', REPO_URL, PROJECT_DIR], check=True)
os.chdir(PROJECT_DIR)
!git pull -q || true
!pip -q install -r requirements.txt
sys.path.insert(0, os.path.join(PROJECT_DIR, 'src'))
import pandas as pd
pd.set_option('display.max_colwidth', 120); pd.set_option('display.width', 200)

Mounted at /content/drive


## 1. Tests (checked against hand-computed values, scikit-learn, and simulated CI coverage)

In [2]:
!python -m pytest -q tests/test_step3_metrics.py

............                                                             [100%]
12 passed in 10.16s


## 2. Demo: three simulated judges on the real RAGTruth labels

In [3]:
import numpy as np
from ljbench import metrics as M
from ljbench.io_utils import read_jsonl
gold = read_jsonl('data/ragtruth_test_600.jsonl')['human_label'].tolist()
rng = np.random.default_rng(0)
flip = {'SUPPORTED': 'UNSUPPORTED', 'UNSUPPORTED': 'SUPPORTED'}
def judge(acc_pos, acc_neg, fail=0.0):
    out = []
    for g in gold:
        if rng.random() < fail: out.append(None); continue
        acc = acc_pos if g == 'UNSUPPORTED' else acc_neg
        out.append(g if rng.random() < acc else flip[g])
    return out
judges = {'strong (85/85)': judge(.85, .85), 'trigger-happy (95/60)': judge(.95, .60),
          'small, 12% broken JSON': judge(.80, .80, fail=.12)}
pd.DataFrame({k: M.summary(gold, v, 'faithfulness') for k, v in judges.items()}).T.round(3)

,n,balanced_acc,ci_lo,ci_hi,kappa,parse_fail,halluc_recall,precision@10%,precision@20%
strong (85/85),600.0,0.873,0.847,0.902,0.747,0.000,0.860,0.457,0.655
trigger-happy (95/60),600.0,0.775,0.744,0.804,0.550,0.000,0.963,0.206,0.368
"small, 12% broken JSON",600.0,0.695,0.660,0.731,0.455,0.118,0.703,0.200,0.359


**Read it like this:** the trigger-happy judge catches 95% of hallucinations, but at a realistic 10% hallucination rate
only about 1 in 5 of its alerts is real (`precision@10%`). Balanced accuracy alone hides that. The broken-JSON judge
loses about 10 points purely to formatting, which is why the parse-failure rate is reported separately.

In [4]:
r = M.paired_bootstrap(gold, judges['strong (85/85)'], judges['trigger-happy (95/60)'])
print('strong - trigger-happy balanced accuracy:', {k: round(v, 3) if isinstance(v, float) else v for k, v in r.items()})

strong - trigger-happy balanced accuracy: {'diff': 0.098, 'lo': 0.061, 'hi': 0.138, 'p_value': 0.0, 'significant': True}


## 3. Demo: position bias on JudgeBench

In [5]:
jb = read_jsonl('data/judgebench_eval_600.jsonl')['human_label'].tolist()
F = {'A': 'B', 'B': 'A'}
def pair_judge(skill, bias):
    # skill: P(knows the right answer); otherwise picks the first slot with prob `bias`
    def pick(g):
        return g if rng.random() < skill else ('A' if rng.random() < bias else 'B')
    orig = [pick(g) for g in jb]
    swap = [pick(F[g]) for g in jb]
    return M.pairwise_outcomes(jb, orig, swap)
pd.DataFrame({'unbiased, skill 0.6': pair_judge(.6, .5), 'first-slot bias 0.9, skill 0.6': pair_judge(.6, .9),
              'coin flip': pair_judge(0, .5)}).T.round(3)

,n_pairs,acc_both_orders,acc_single_order,position_consistency,share_choosing_first
"unbiased, skill 0.6",600.0,0.658,0.817,0.683,0.472
"first-slot bias 0.9, skill 0.6",600.0,0.590,0.788,0.603,0.668
coin flip,600.0,0.227,0.495,0.463,0.477


## Result and explanation

Expected: **12 passed.**

- `acc_both_orders` is the headline Task B number: a pair counts only if the judge picks the same response in both orders **and** it is the right one.
  A coin flip scores about 23% on it vs 50% single-order. In the demo, a first-slot bias costs about 7 points on `acc_both_orders` but only about 3 points on single-order accuracy, so single-order accuracy hides position bias.
- The bootstrap CI covered the true accuracy in 88–99% of simulated runs (test), and the paired bootstrap only flags real differences.
- **At n = 600 the 95% CI is about ±3 points** (demo: 0.847–0.902), so two judges a few points apart should be reported as tied unless the paired bootstrap says otherwise.

**Gate passed → Step 4 (API runner).**